Data Preparation

In [1]:
import pandas as pd 
import re

In [2]:
# load data 
df = pd.read_csv("cleaned_data/bangalore_cleaned.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170,2.0,1.0,38.00
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785,5.0,3.0,295.00


In [3]:
df.isnull().sum()

area_type       0
availability    0
location        0
size            0
society         0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


size is object type and total_sqft is also object type.

We will convert total_sqft to numeric and for different formats present in this feature eg 2100 - 2200 , we will take the average of the two numbers and convert the entire column to numeric type.

We will Extract BHK from the size column and this feature also has two unique formats eg. 2 BHK , 2 Bedroom. We will consider both of them as 2 BHK and convert the entire column to numeric type.

In [5]:
# function to convert the total_sqft column to numeric

def convert_sqft(x):
    try:
        s = str(x).strip()
        if "-" in s:
            parts = s.split("-")
            return (float(parts[0]) + float(parts[1]))/2
        return float(s)

    except: # if parsing fails, return None
        return None

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [6]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785.0,5.0,3.0,295.00


In [7]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft      15
bath             0
balcony          0
price            0
dtype: int64

In [8]:
# Dropping the rows with null values in total_sqft column
df = df.dropna(subset=["total_sqft"])

In [9]:
# Function to extract BHK from the size column

def extract_bhk(x):
    if pd.isna(x): return None
    m = re.search(r"(\d+)\s*(bhk|bedroom)" , str(x).lower())

    if m:
        return int(m.group(1)) # return the number part as integer

    return None


df['bhk'] = df['size'].apply(extract_bhk)

In [10]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00,4.0
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00,3.0
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00,2.0
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785.0,5.0,3.0,295.00,4.0


In [11]:
df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft       0
bath             0
balcony          0
price            0
bhk             10
dtype: int64

In [12]:
# drop the rows with null values in bhk column
df = df.dropna(subset=["bhk"])

In [13]:
# drop size column as we have extracted bhk from it

df = df.drop("size" , axis=1)

In [14]:
# Create a new column to provide descriptive text for each property listing

df['text'] = (
    df['bhk'].astype(int).astype(str) + " BHK property in " + df['location'] +
    " with " + df['total_sqft'].astype(int).astype(str) + " sqft, priced at " +
    df['price'].astype(str) + " lakh. " +
    "Bathrooms: " + df['bath'].astype(str) + ". " +
    "Balconies: " + df['balcony'].astype(str) + ". " +
    "Area type: " + df['area_type'] + ". " +
    "Availability: " + df['availability'] + ". " +
    "Society: " + df['society']
)

In [15]:
df.head()

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
0,Super built-up Area,19-Dec,Electronic City Phase II,Coomee,1056.0,2.0,1.0,39.07,2.0,2 BHK property in Electronic City Phase II wit...
1,Plot Area,Ready To Move,Chikka Tirupathi,Theanmp,2600.0,5.0,3.0,120.00,4.0,4 BHK property in Chikka Tirupathi with 2600 s...
2,Super built-up Area,Ready To Move,Lingadheeranahalli,Soiewre,1521.0,3.0,1.0,95.00,3.0,3 BHK property in Lingadheeranahalli with 1521...
3,Super built-up Area,Ready To Move,Whitefield,DuenaTa,1170.0,2.0,1.0,38.00,2.0,"2 BHK property in Whitefield with 1170 sqft, p..."
4,Plot Area,Ready To Move,Whitefield,Prrry M,2785.0,5.0,3.0,295.00,4.0,"4 BHK property in Whitefield with 2785 sqft, p..."


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7119 entries, 0 to 7143
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7119 non-null   object 
 1   availability  7119 non-null   object 
 2   location      7119 non-null   object 
 3   society       7119 non-null   object 
 4   total_sqft    7119 non-null   float64
 5   bath          7119 non-null   float64
 6   balcony       7119 non-null   float64
 7   price         7119 non-null   float64
 8   bhk           7119 non-null   float64
 9   text          7119 non-null   object 
dtypes: float64(5), object(5)
memory usage: 611.8+ KB


In [19]:
# find the maxumum length of the text column
max_length = df['text'].apply(len).max()
print("Maximum length of text column:", max_length)

Maximum length of text column: 197


In [ ]:
# Creating embedding for each row in the text column using Hugging Face's sentence-transformers library

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss 


# Prepare texts and IDs 

texts = df['text'].tolist()
ids = np.arange(len(texts)).astype('int64')

# Create embeddings 

embed_model = SentenceTransformer('all-MiniLM-L6-v2') # small and fast for generating embeddings

embeddings = embed_model.encode(texts,show_progress_bar=True, convert_to_numpy=True)

# Normalize the embeddings (This helps with the cosine similarity search)

faiss.normalize_L2(embeddings)

# Build Faiss index 

d = embeddings.shape[1] # dimension of the embeddings
cpu_index = faiss.IndexFlatIP(d) # inner product on normalized vectors is equivalent to cosine similarity

# move to gpu 
res = faiss.StandardGpuResources() # Initialize GPU resources
index = faiss.index_cpu_to_gpu(res, 0, cpu_index) # move the index to GPU


index.add(embeddings) # add the embeddings to the index

print(f"Index is on GPU: {index.is_trained} and has {index.ntotal} vectors.")

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ankur Bhatt\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 710.68i

Index is on GPU: True and has 7119 vectors.


In [24]:
# Retrival Function

def retrieve(query , top_k=5):
    q_emb = embed_model.encode([query] , convert_to_numpy=True) 
    faiss.normalize_L2(q_emb) # normalize the query embedding
    D, I = index.search(q_emb , top_k) # search the index for the top_k most similar vectors
    results = []

    for idx , score in zip(I[0],D[0]):
        if idx == -1: # if no more results
            continue 
        results.append({'id':int(idx) ,
                         'score' : float(score) ,
                           'text' : texts[int(idx)]
                           }
                      )
        
    return results

In [32]:
import os 
os.environ['HF_HOME'] = "D:/hf_cache"

In [33]:
# Simple response generation using retrieved context : small Hugging Face model - google/flan-t5-small
from transformers import pipeline , AutoTokenizer , AutoModelForSeq2SeqLM

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline('text-generation' ,
               model =  model_id ,
                 device = 0 ,
                 dtype = 'auto',
                 model_kwargs = {'cache_dir': "D:/hf_cache"})




def answer_query(query , top_k = 5 , max_new_tokens = 300):
    retrieved = retrieve(query , top_k = top_k)

    # Build a concise context string from top results

    context = "\n\n".join([f"Listing {r['id']} : {r['text']}" for r in retrieved])

    prompt = f"System: You are a helpful real estate assistant. Answer based ONLY on the context provided.\n\nContext:\n{context}\n\nUser: {query}\n\nAssistant:"

    out = gen(prompt , max_new_tokens = max_new_tokens , do_sample = False , return_full_text = False)
    # return full text = False ensures we only get the generated answer without the prompt
    return {'response':out[0]['generated_text'].strip() , 'retrieved':retrieved}





d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ankur Bhatt\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packa

In [34]:
# Example queries and display result

example_queries = [
    "Show me 3 BHK properties in Whitefield under 80 lakhs",
    "Find 2 BHK apartments in Koramangala with at least 1000 sqft area and price under 1.2 crore",
    "List affordable 1 BHK options in Jayanagar under 50 lakhs"
]

def pretty_print_retrieved(retrieved , df , show_full_row = False):
    print("\nTop retrieved listings:")
    for r in retrieved:
        print(f"  id={r['id']}  score={r['score']:.4f}")
        print(f"  snippet: {r['text']}")
    if show_full_row and len(retrieved)>0:
        print("\nFull dataframe rows for retrieved listings:")
        ids = [r['id'] for r in retrieved]
        display(df.iloc[ids]) # This renders a table in jupyter notebook



In [35]:
for q in example_queries:
    print("\n" + "="*80)
    print("Query:",q)
    try:
        res = answer_query(q,top_k = 5 , max_new_tokens =200)
    except Exception as e:
        print("Error running answer_query:",e)
        continue
    print("\n--- Generated Response ---")
    print(res['response'])
    pretty_print_retrieved(res['retrieved'] , df , show_full_row = True)


Query: Show me 3 BHK properties in Whitefield under 80 lakhs


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated Response ---
Based on the information provided, here are three 3-BHK properties in Whitefield that meet or fall below the price limit of 80 Lakhs:

1. **Listing 4442** - This listing offers a 3-BHK property with 3450 square feet and is priced at 208.0 Lakhs.
   
2. **Listing 4602** - The 3-BHK property in this listing has 1275 square feet and is priced at 65.0 Lakhs.

3. **Listing 4103** - This listing provides a 3-BHK property with 1537 square feet and is priced at 70.0 Lakhs.

These listings all match your criteria for being 3-BHK properties located in Whitefield and having a price under 80 Lakhs. If you need more details or have any specific requirements, feel free to ask!

Top retrieved listings:
  id=4442  score=0.7540
  snippet: 3 BHK property in Whitefield with 3450 sqft, priced at 208.0 lakh. Bathrooms: 4.0. Balconies: 2.0. Area type: Plot  Area. Availability: Ready To Move. Society: StodsWa
  id=4602  score=0.7441
  snippet: 3 BHK property in Whitefield with 127

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
4461,Plot Area,Ready To Move,Whitefield,StodsWa,3450.0,4.0,2.0,208.0,3.0,"3 BHK property in Whitefield with 3450 sqft, p..."
4622,Built-up Area,Ready To Move,Whitefield,Laave E,1275.0,2.0,3.0,65.0,3.0,"3 BHK property in Whitefield with 1275 sqft, p..."
3753,Super built-up Area,Ready To Move,Whitefield,UKe 2nz,2225.0,3.0,1.0,149.0,3.0,"3 BHK property in Whitefield with 2225 sqft, p..."
4121,Super built-up Area,Ready To Move,Whitefield,BMttaor,1537.0,2.0,3.0,70.0,3.0,"3 BHK property in Whitefield with 1537 sqft, p..."
535,Super built-up Area,Ready To Move,Whitefield,UKe 2nz,2225.0,2.0,1.0,125.0,3.0,"3 BHK property in Whitefield with 2225 sqft, p..."


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: Find 2 BHK apartments in Koramangala with at least 1000 sqft area and price under 1.2 crore

--- Generated Response ---
Based on the criteria you've specified (2 BHK apartments in Koramangala with an area of at least 1000 sqft and a price under 1.2 crore), here is the list that meets these requirements:

- **Listing 558**: This listing has a 2 BHK apartment in Koramangala with 1005 sqft, priced at 110.0 lakh.
- **Listing 1024**: This listing also has a 2 BHK apartment in Koramangala with 1300 sqft, priced at 110.0 lakh.

Both listings meet all the given conditions. If there's any additional information or preference for further filtering, please let me know!

Top retrieved listings:
  id=558  score=0.7984
  snippet: 2 BHK property in Koramangala with 1005 sqft, priced at 110.0 lakh. Bathrooms: 1.0. Balconies: 1.0. Area type: Super built-up  Area. Availability: Ready To Move. Society: K ncy R
  id=1024  score=0.7974
  snippet: 2 BHK property in Koramangala with 1300 sqft, priced

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
560,Super built-up Area,Ready To Move,Koramangala,K ncy R,1005.0,1.0,1.0,110.0,2.0,"2 BHK property in Koramangala with 1005 sqft, ..."
1029,Super built-up Area,Ready To Move,Koramangala,Saace P,1300.0,2.0,3.0,110.0,2.0,"2 BHK property in Koramangala with 1300 sqft, ..."
3190,Super built-up Area,Ready To Move,Koramangala,Jainaht,1300.0,2.0,2.0,82.0,2.0,"2 BHK property in Koramangala with 1300 sqft, ..."
1772,Built-up Area,Ready To Move,Koramangala,Proodt,1900.0,2.0,1.0,200.0,2.0,"2 BHK property in Koramangala with 1900 sqft, ..."
2584,Super built-up Area,Ready To Move,Koramangala,Proodt,1320.0,2.0,1.0,150.0,2.0,"2 BHK property in Koramangala with 1320 sqft, ..."


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: List affordable 1 BHK options in Jayanagar under 50 lakhs

--- Generated Response ---
Based on the information provided, there is no listing for an affordable 1 BHK option in Jayanagar under 50 lakhs. The available listings range from 1170 sqft to 1875 sqft and prices start from 105.0 lakh up to 235.0 lakh. Therefore, none of these properties meet both criteria (affordability and size) simultaneously as per your request. If you have any other questions or need further assistance, feel free to ask!

Top retrieved listings:
  id=6405  score=0.7812
  snippet: 3 BHK property in Jayanagar with 2240 sqft, priced at 300.0 lakh. Bathrooms: 3.0. Balconies: 2.0. Area type: Super built-up  Area. Availability: 18-May. Society: PrianM 
  id=1365  score=0.7753
  snippet: 2 BHK property in Jayanagar with 1170 sqft, priced at 105.0 lakh. Bathrooms: 2.0. Balconies: 3.0. Area type: Super built-up  Area. Availability: Ready To Move. Society: Rakulai
  id=3857  score=0.7721
  snippet: 3 BHK proper

,area_type,availability,location,society,total_sqft,bath,balcony,price,bhk,text
6428,Super built-up Area,18-May,Jayanagar,PrianM,2240.0,3.0,2.0,300.0,3.0,"3 BHK property in Jayanagar with 2240 sqft, pr..."
1371,Super built-up Area,Ready To Move,Jayanagar,Rakulai,1170.0,2.0,3.0,105.0,2.0,"2 BHK property in Jayanagar with 1170 sqft, pr..."
3875,Built-up Area,15-Dec,1st Block Jayanagar,Bhsic C,1200.0,2.0,0.0,130.0,3.0,3 BHK property in 1st Block Jayanagar with 120...
1601,Built-up Area,Ready To Move,Jayanagar,Rathira,1875.0,2.0,3.0,235.0,3.0,"3 BHK property in Jayanagar with 1875 sqft, pr..."
4741,Super built-up Area,Ready To Move,7th Block Jayanagar,Jaitira,1707.0,3.0,2.0,171.0,3.0,3 BHK property in 7th Block Jayanagar with 170...
